In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/thapaprabin4/insurance1/insurance.csv
/kaggle/input/datasets/thapaprabin4/insurance/insurancepractice


In [2]:
import pandas as pd 
df=pd.read_csv(r'/kaggle/input/datasets/thapaprabin4/insurance1/insurance.csv')
df.head()

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


Empirical and marginal probability

In [3]:
p_smoker=(df['smoker']=='yes').mean()
print(p_smoker)
#a random selected individual have probability of 20% to be a smoker


0.20478325859491778


In [4]:
p_southeast = (df['region'] == 'southeast').mean()
print(f"P(Region = 'southeast'): {p_southeast:.4f}")

P(Region = 'southeast'): 0.2720


In [5]:
p_southwest = (df['region'] == 'southwest').mean()
print(f"P(Region = 'southwest'): {p_southwest:.4f}")

P(Region = 'southwest'): 0.2429


In [6]:
p_northeast = (df['region'] == 'northeast').mean()
print(f"P(Region = 'northeast'): {p_northeast:.4f}")

P(Region = 'northeast'): 0.2422


In [7]:
p_northwest = (df['region'] == 'northwest').mean()
print(f"P(Region = 'northwest'): {p_northwest:.4f}")

P(Region = 'northwest'): 0.2429


In [8]:
'''The policyholders are almost evenly balanced across
the four US geographic regions (~25% per region), with a 
slight overrepresentation in the Southeast (~27.2%).'''

'The policyholders are almost evenly balanced across\nthe four US geographic regions (~25% per region), with a \nslight overrepresentation in the Southeast (~27.2%).'

In [9]:
p_obese=(df['bmi']>=30).mean()
print(p_obese)

#Over half of the insured population is clinically obese:52%

0.5284005979073244


In [10]:
#probability that policy holder have no children
p_no_child=(df['children']==0).mean()
print(p_no_child)

0.4289985052316891


In [11]:
#hypothesis testing 
#if average age of obese people greater than 30?
#H0:age of obese people <= 30yrs
#H1: age of stroke patient is >30yyears
filter_condition=df['bmi']>=30
obese=df[filter_condition]
obese_age=obese['age']

In [12]:
from statsmodels.stats.weightstats import ztest
z_stat,p_value=ztest(obese_age,value=30,alternative='larger')

if p_value > 0.05:
    print("Age of obese people less than 30 years")
    
else:
    print('Age of obese people is greater than 30 years')

Age of obese people is greater than 30 years


In [13]:
#based on policy holders do males have lower average bmi than female
#h0:male_bmi >= female_bmi
#h1:male_bmi < female_bmi
male_bmi=df[df['sex']=='male']['bmi']
female_bmi=df[df['sex']=='female']['bmi']
z_stat,p_value=ztest(male_bmi,female_bmi,alternative='smaller')
if p_value > 0.05:
    print("male bmi is greater than female bmi")
    
else:
    print('female bmi is greater than male bmi')

male bmi is greater than female bmi


** T-TEST**


In [14]:
filter_condition=df['bmi']>=30
obese=df[filter_condition]
obese_age=obese['age'][0:28]
print(len(obese_age))

28


In [15]:

from scipy.stats import ttest_1samp
t_stat,p_value=ttest_1samp(obese_age,30,alternative='less')
if p_value>0.05:
    print('Avg age greater than or equal to 30yrs')
else:
    print("Avg age is less than 30yrs.")

Avg age greater than or equal to 30yrs


CHI SQUARED TEST 


In [16]:
#IS SMOKER RELATED TO AGE 
TABLE=pd.crosstab(df['smoker'],df['age'])
TABLE

age,18,19,20,21,22,23,24,25,26,27,...,55,56,57,58,59,60,61,62,63,64
smoker,,,,,,,,,,,,,,,,,,,,,
no,57,50,20,26,22,21,22,23,25,19,...,24,22,22,24,21,18,17,19,18,15
yes,12,18,9,2,6,7,6,5,3,9,...,2,4,4,1,4,5,6,4,5,7


In [17]:
from scipy.stats import chi2_contingency
chi2_stat,p_value,dof,expected=chi2_contingency(TABLE)
if p_value > 0.05:
    
    print('no relation ')
else:
    print('smoker and age are related')

no relation 


In [18]:

table=pd.crosstab(df['bmi'],df['age'])
table
from scipy.stats import chi2_contingency
chi2_stat,p_value,dof,expected=chi2_contingency(TABLE)
if p_value > 0.05:
    
    print('no relation ')
else:
    print('bmi and age are related')

no relation 


ANOVA

In [19]:
import pandas as pd
import scipy.stats as stats

# 1. Bin continuous age into categorical groups
bins = [17, 29, 49, 100]
labels = ['Young (18-29)', 'Middle (30-49)', 'Senior (50+)']
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels)

# 2. Display summary statistics per age group
bmi_summary = df.groupby('age_group')['bmi'].agg(['count', 'mean', 'std'])
print("--- Mean BMI by Age Group ---")
print(bmi_summary)
print("\n" + "="*40 + "\n")

# 3. Extract BMI values for each group
group_young = df[df['age_group'] == 'Young (18-29)']['bmi']
group_middle = df[df['age_group'] == 'Middle (30-49)']['bmi']
group_senior = df[df['age_group'] == 'Senior (50+)']['bmi']

# 4. Run One-Way ANOVA test
f_stat, p_val = stats.f_oneway(group_young, group_middle, group_senior)

print(f"ANOVA F-Statistic: {f_stat:.4f}")
print(f"p-value:           {p_val:.4e}")

# 5. Statistical Verdict (Alpha = 0.05)
if p_val < 0.05:
    print("\nResult: Reject H0 — Significant difference in average BMI exists across age groups.")
else:
    print("\nResult: Fail to reject H0 — No significant difference in average BMI across age groups.")

--- Mean BMI by Age Group ---
                count       mean       std
age_group                                 
Young (18-29)     417  29.847590  6.229227
Middle (30-49)    536  30.582192  5.991059
Senior (50+)      385  31.660065  5.975622


ANOVA F-Statistic: 9.0281
p-value:           1.2747e-04

Result: Reject H0 — Significant difference in average BMI exists across age groups.


/tmp/ipykernel_16/401349361.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  bmi_summary = df.groupby('age_group')['bmi'].agg(['count', 'mean', 'std'])
